# Etapa 2 — Preparação e integração dos dados

**Integrantes:** Daniel Suzuki Naves (RA 2363062); João Pedro Novais Zanusso (RA 2607743); Kalil Nege (RA 2409020).

Interrupções na distribuição da Copel no Paraná, carga elétrica, clima e feriados.

Recorte: 01/01/2022 a 16/11/2026. O ano de 2026 é parcial. Este notebook contém o download, a limpeza, as agregações, a integração e as verificações.

## 1. Ambiente e arquivos

Execute as células em ordem. Na primeira execução, arquivos ausentes serão baixados dos portais oficiais. O processamento da ANEEL lê arquivos nacionais e pode levar alguns minutos.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures
import hashlib
import json
import platform
import time
import unicodedata

import holidays
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import requests
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'requirements.txt').is_file() and (p / 'docs/especificacao_projeto.pdf').is_file())
RAW = ROOT / 'data/raw'
OUT = ROOT / 'data/processed'
for pasta in [RAW, OUT]:
    pasta.mkdir(parents=True, exist_ok=True)
CNPJ = '04368898000106'
YEARS = range(2022, 2027)
END = '2026-11-16'
PONTOS = {
    'curitiba': (-25.43, -49.27), 'londrina': (-23.31, -51.16),
    'maringa': (-23.42, -51.94), 'cascavel': (-24.96, -53.46),
    'ponta_grossa': (-25.10, -50.16),
}
quality = {}
schemas = {}
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 65)
print(f'Corte: {END} | Copel Distribuição: CNPJ {CNPJ}')

Corte: 2026-11-16 | Copel Distribuição: CNPJ 04368898000106


In [3]:
def baixar_arquivo(job):
    url, destino = job
    if destino.exists():
        return
    parcial = destino.with_suffix(destino.suffix + '.part')
    for tentativa in range(4):
        try:
            with requests.get(url, timeout=(20, 60), stream=True) as resposta:
                resposta.raise_for_status()
                with parcial.open('wb') as arquivo:
                    for bloco in resposta.iter_content(1024 * 1024):
                        arquivo.write(bloco)
            parcial.replace(destino)
            return
        except requests.RequestException:
            if tentativa == 3:
                raise
            time.sleep(2 ** tentativa)


def baixar_originais():
    catalogos = {
        'interrupcoes': ('https://dadosabertos.aneel.gov.br', 'interrupcoes-de-energia-eletrica-nas-redes-de-distribuicao'),
        'ocorrencias': ('https://dadosabertos.aneel.gov.br', 'ocorrencias-emergenciais-nas-redes-de-distribuicao'),
        'ons': ('https://dados.ons.org.br', 'curva-carga'),
    }
    jobs = []
    for nome, (host, identificador) in catalogos.items():
        caminho = RAW / f'{nome}_catalogo.json'
        if caminho.exists():
            catalogo = json.loads(caminho.read_text())
        else:
            resposta = requests.get(host + '/api/3/action/package_show', params={'id': identificador}, timeout=60)
            resposta.raise_for_status()
            catalogo = resposta.json()
            caminho.write_text(json.dumps(catalogo, ensure_ascii=False, indent=2))
        for recurso in catalogo['result']['resources']:
            url = recurso['url']
            ano_selecionado = any(str(ano) in url for ano in YEARS)
            dados = ano_selecionado and url.endswith('.csv' if nome == 'ons' else '.parquet')
            dicionario = url.endswith('.pdf') and (nome == 'ons' or any(ano in recurso['name'] for ano in ['2025', '2026']))
            if dados or dicionario:
                prefixo = '_novo_' if dicionario and '2026' in recurso['name'] else '_'
                jobs.append((url, RAW / (nome + prefixo + url.split('/')[-1])))
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        list(executor.map(baixar_arquivo, jobs))

    for nome, (lat, lon) in PONTOS.items():
        caminho = RAW / f'clima_{nome}_2026.json'
        if caminho.exists():
            continue
        parametros = dict(latitude=lat, longitude=lon, start_date='2022-01-01', end_date=END,
                          daily='temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_gusts_10m_max',
                          timezone='America/Sao_Paulo', models='era5', wind_speed_unit='kmh')
        resposta = requests.get('https://archive-api.open-meteo.com/v1/archive', params=parametros, timeout=120)
        resposta.raise_for_status()
        dados = resposta.json()
        dados['_request_url'] = resposta.url
        parcial = caminho.with_suffix('.json.part')
        parcial.write_text(json.dumps(dados, ensure_ascii=False, indent=2))
        parcial.replace(caminho)
        time.sleep(1)
    print('Originais disponíveis.')

baixar_originais()

Originais disponíveis.


## 2. Fontes e esquemas

A ANEEL fornece duas bases de distribuição: interrupções e ocorrências emergenciais. Elas permanecem separadas. A seleção da distribuidora usa CNPJ. A carga horária do ONS cobre o **subsistema Sul**, não apenas o Paraná.

Em 2026, a ANEEL alterou campos e identificadores. O adaptador abaixo mantém a identificação do esquema nos microdados.

In [4]:
fontes = []
for nome in ['interrupcoes', 'ocorrencias', 'ons']:
    catalogo = json.loads((RAW / f'{nome}_catalogo.json').read_text())['result']
    fontes.append({
        'fonte': nome,
        'conjunto': catalogo['title'],
        'recursos_no_catalogo': len(catalogo['resources']),
    })
display(pd.DataFrame(fontes))

mudancas = []
for fonte in ['interrupcoes', 'ocorrencias']:
    esquemas = {}
    for ano in [2025, 2026]:
        arquivo = next(RAW.glob(f'{fonte}_*{ano}.parquet'))
        esquemas[ano] = pq.ParquetFile(arquivo).schema_arrow
    antigos = set(esquemas[2025].names)
    novos = set(esquemas[2026].names)
    mudancas.append({
        'base': fonte,
        'campos_2025': len(antigos),
        'campos_2026': len(novos),
        'apenas_2025': ', '.join(sorted(antigos - novos)),
        'apenas_2026': ', '.join(sorted(novos - antigos)),
    })
display(pd.DataFrame(mudancas))
display(pd.read_csv(RAW / 'ons_CURVA_CARGA_2026.csv', sep=';', nrows=4))
for fonte in ['interrupcoes', 'ocorrencias']:
    for ano in [2025, 2026]:
        arquivo = next(RAW.glob(f'{fonte}_*{ano}.parquet'))
        print(f'\n{fonte} — {ano}')
        print(pq.ParquetFile(arquivo).schema_arrow)

,fonte,conjunto,recursos_no_catalogo
0,interrupcoes,Interrupções de Energia Elétrica nas Redes de Distribuição,22
1,ocorrencias,Ocorrências Emergenciais nas Redes de Distribuição,22
2,ons,Curva de Carga Horária,83


,base,campos_2025,campos_2026,apenas_2025,apenas_2026
0,interrupcoes,18,26,"DscAlimentadorSubestacao, DscFatoGeradorInterrupcao, DscSubes...","AnoCompetencia, CodAlimentador, CodConjUnidadeConsumidora, Co..."
1,ocorrencias,15,23,"CodIBGE, DscCanalAtendimento, DscNumInterrupcao, DscOcorrenci...","CodAlimentador, CodConjUnidConsumidora, CodMunicipioIBGE, Cod..."


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
0,N,NORTE,2026-01-01 00:00:00,7622.606
1,NE,NORDESTE,2026-01-01 00:00:00,12887.229
2,S,SUL,2026-01-01 00:00:00,11137.516
3,SE,SUDESTE/CENTRO-OESTE,2026-01-01 00:00:00,41048.460



interrupcoes — 2025
DatGeracaoConjuntoDados: date32[day]
IdeConjuntoUnidadeConsumidora: int64
DscConjuntoUnidadeConsumidora: string
DscAlimentadorSubestacao: string
DscSubestacaoDistribuicao: string
NumOrdemInterrupcao: string
DscTipoInterrupcao: string
IdeMotivoInterrupcao: int64
DatInicioInterrupcao: timestamp[ms]
DatFimInterrupcao: timestamp[ms]
DscFatoGeradorInterrupcao: string
NumNivelTensao: int64
NumUnidadeConsumidora: int64
NumConsumidorConjunto: int64
NumAno: int64
NomAgenteRegulado: string
SigAgente: string
NumCPFCNPJ: int64

interrupcoes — 2026
DatGeracaoConjuntoDados: string
NumCNPJDistribuidora: string
NomAgente: string
SigAgente: string
CodMunicipioIBGE: string
CodInterrupcao: string
CodEvento: string
CodOcorrencia: string
CodConjUnidadeConsumidora: string
DscConjuntoUnidadeConsumidora: string
CodAlimentador: string
CodSubestacao: string
AnoCompetencia: string
MesCompetencia: string
DscLocalizacaoInterrupcao: string
DscMotivoExpurgo: string
DatInicioInterrupcao: string
D

## 3. Limpeza e preparação

São removidas apenas duplicatas exatas. Registros com a mesma chave e conteúdo diferente permanecem para revisão. Datas de início inválidas não entram na série diária; durações e quantidades inválidas ficam fora das respectivas medidas. Eventos acima de sete dias são sinalizados.

As funções abaixo filtram a distribuidora, harmonizam os esquemas e preparam cada fonte.

In [5]:
def save(df, name):
    OUT.mkdir(exist_ok=True, parents=True)
    path = OUT / f'{name}.parquet'
    if name.endswith('_revisao') and df.empty:
        path.unlink(missing_ok=True)
        return
    df.to_parquet(path, index=False)

def cnpj(s):
    return s.astype('string').str.replace('\\.0$', '', regex=True).str.replace('\\D', '', regex=True).str.zfill(14)

def require(df, cols, name):
    missing = set(cols) - set(df.columns)
    if missing:
        raise ValueError(f'{name}: esquema inesperado: {missing}')

def aneel(kind):
    frames = []
    for year in YEARS:
        paths = list(RAW.glob(f'{kind}_*{year}.parquet'))
        if len(paths) != 1:
            raise ValueError(f'Arquivo original ausente/ambíguo: {kind} {year}')
        f = pq.ParquetFile(paths[0])
        schemas[paths[0].name] = str(f.schema_arrow)
        cnpj_col = next((c for c in ['NumCPFCNPJ', 'NumCNPJDistribuidora', 'NumCnpjDistribuidora'] if c in f.schema_arrow.names), None)
        if cnpj_col is None:
            raise ValueError('CNPJ não identificado')
        kept = []
        for batch in f.iter_batches(batch_size=150000):
            codes = pc.utf8_lpad(pc.utf8_trim_whitespace(pc.cast(batch[cnpj_col], pa.string())), width=14, padding='0')
            selected = batch.filter(pc.equal(codes, CNPJ))
            if selected.num_rows == 0:
                continue
            d = selected.to_pandas()
            d = d.loc[cnpj(d[cnpj_col]).eq(CNPJ)].copy()
            if len(d):
                kept.append(d)
        if not kept:
            raise ValueError(f'Copel Distribuição não encontrada: {kind} {year}')
        d = pd.concat(kept, ignore_index=True)
        d['NumCPFCNPJ'] = cnpj(d[cnpj_col])
        d['esquema_aneel'] = '2026' if year == 2026 else 'ate2025'
        if year == 2026:
            if kind == 'interrupcoes':
                mapping = {'CodInterrupcao': 'NumOrdemInterrupcao', 'CodConjUnidadeConsumidora': 'IdeConjuntoUnidadeConsumidora', 'QtdConsumidoresAfetados': 'NumUnidadeConsumidora', 'QtdConsumidoresAtivos': 'NumConsumidorConjunto', 'AnoCompetencia': 'NumAno', 'DscFatoGeradorTipo': 'DscTipoInterrupcao'}
                for (source, target) in mapping.items():
                    d[target] = d[source]
                cols = ['DscFatoGeradorOrigem', 'DscFatoGeradorTipo', 'DscFatoGeradorCausa', 'DscFatoGeradorDetalhe']
                d['DscFatoGeradorInterrupcao'] = d[cols[0]].fillna('').astype('string').str.cat([d[c].fillna('').astype('string') for c in cols[1:]], sep=' - ')
            else:
                mapping = {'CodOcorrenciaEmergencial': 'NumOcorrencia', 'DthIniOcorrencia': 'DthInicioOcorrenciaAberta', 'DthFimOcorrencia': 'DthFimOcorrenciaAberta', 'CodMunicipioIBGE': 'CodIBGE', 'NumTempoPreparacao': 'MdaPreparo', 'NumTempoDeslocamento': 'MdaDeslocamento', 'NumTempoExecucao': 'MdaExecucao'}
                for (source, target) in mapping.items():
                    d[target] = d[source]
        for c in ['NumAno', 'IdeConjuntoUnidadeConsumidora', 'NumUnidadeConsumidora', 'NumConsumidorConjunto', 'CodIBGE', 'NumNivelTensao']:
            if c in d:
                d[c] = pd.to_numeric(d[c], errors='coerce').astype('Int64')
        for c in ['NumOrdemInterrupcao', 'NumOcorrencia']:
            if c in d:
                d[c] = d[c].astype('string')
        for c in ['DatGeracaoConjuntoDados', 'DatInicioInterrupcao', 'DatFimInterrupcao', 'DthInicioOcorrenciaAberta', 'DthFimOcorrenciaAberta']:
            if c in d:
                d[c] = pd.to_datetime(d[c], errors='coerce')
        for c in ['MdaPreparo', 'MdaDeslocamento', 'MdaExecucao']:
            if c in d:
                d[c] = pd.to_numeric(d[c].astype('string').str.replace(',', '.', regex=False), errors='coerce')
        d['arquivo_origem'] = paths[0].name
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    before = len(df)
    subset = [c for c in df if c not in ['DatGeracaoConjuntoDados', 'arquivo_origem']]
    dup = df.duplicated(subset=subset)
    df = df.loc[~dup].copy()
    quality[kind] = {'linhas_copel_originais': before, 'duplicados_exatos': int(dup.sum()), 'linhas_sem_duplicados': len(df)}
    return df

In [6]:
def interrupcoes():
    df = aneel('interrupcoes')
    require(df, ['DatInicioInterrupcao', 'DatFimInterrupcao', 'NumOrdemInterrupcao', 'IdeConjuntoUnidadeConsumidora', 'DscFatoGeradorInterrupcao', 'NumUnidadeConsumidora', 'DscTipoInterrupcao'], 'interrupcoes')
    for c in ['DatInicioInterrupcao', 'DatFimInterrupcao']:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    df['duracao_h'] = (df.DatFimInterrupcao - df.DatInicioInterrupcao).dt.total_seconds() / 3600
    df['data'] = df.DatInicioInterrupcao.dt.normalize()
    df['inicio_invalido'] = df.data.isna() | ~df.data.between('2022-01-01', END)
    df['duracao_invalida'] = df.duracao_h.isna() | df.duracao_h.lt(0)
    df['uc_invalida'] = df.NumUnidadeConsumidora.isna() | df.NumUnidadeConsumidora.lt(0)
    df['duracao_suspeita_7dias'] = df.duracao_h.gt(168)
    quality['interrupcoes'].update({c: int(df[c].sum()) for c in ['inicio_invalido', 'duracao_invalida', 'uc_invalida', 'duracao_suspeita_7dias']})
    save(df.loc[df.inicio_invalido | df.duracao_invalida | df.uc_invalida | df.duracao_suspeita_7dias], 'interrupcoes_revisao')
    df = df.loc[~df.inicio_invalido].copy()
    df['duracao_h_valida'] = df.duracao_h.mask(df.duracao_invalida)
    df['uc_eventos'] = df.NumUnidadeConsumidora.mask(df.uc_invalida)
    df['uc_horas'] = df.uc_eventos * df.duracao_h_valida
    df['nao_programada'] = df.DscTipoInterrupcao.astype('string').fillna('').map(lambda x: ''.join((c for c in unicodedata.normalize('NFD', x.casefold().strip()) if unicodedata.category(c) != 'Mn'))).eq('nao programada')
    df['id_interrupcao'] = df.NumAno.astype('string') + '|' + df.NumOrdemInterrupcao.astype('string').str.strip()
    key = ['NumCPFCNPJ', 'NumAno', 'NumOrdemInterrupcao', 'IdeConjuntoUnidadeConsumidora']
    conflicts = df.duplicated(key, keep=False)
    quality['interrupcoes']['linhas_chave_repetida'] = int(conflicts.sum())
    save(df, 'interrupcoes_copel')
    g = df.groupby('data')
    daily = g.agg(n_registros_interrupcao=('id_interrupcao', 'size'), n_ids_interrupcao=('id_interrupcao', 'nunique'), n_nao_programadas=('nao_programada', 'sum'), n_duracoes_validas=('duracao_h_valida', 'count'), duracao_media_h=('duracao_h_valida', 'mean'))
    for (src, dst) in [('uc_eventos', 'uc_eventos_soma'), ('uc_horas', 'uc_horas_soma')]:
        daily[dst] = g[src].sum(min_count=1)
    save(df.groupby(['data', 'DscFatoGeradorInterrupcao'], dropna=False).size().reset_index(name='n_registros'), 'fatos_geradores_diarios')
    return daily

In [7]:
def ocorrencias():
    df = aneel('ocorrencias')
    require(df, ['NumOcorrencia', 'DthInicioOcorrenciaAberta', 'DthFimOcorrenciaAberta'], 'ocorrencias')
    for c in ['DthInicioOcorrenciaAberta', 'DthFimOcorrenciaAberta']:
        df[c] = pd.to_datetime(df[c], errors='coerce')
    df['data'] = df.DthInicioOcorrenciaAberta.dt.normalize()
    df['duracao_atendimento_h'] = (df.DthFimOcorrenciaAberta - df.DthInicioOcorrenciaAberta).dt.total_seconds() / 3600
    df['data_invalida'] = df.data.isna() | ~df.data.between('2022-01-01', END)
    df['duracao_invalida'] = df.duracao_atendimento_h.isna() | df.duracao_atendimento_h.lt(0)
    quality['ocorrencias'].update({c: int(df[c].sum()) for c in ['data_invalida', 'duracao_invalida']})
    save(df.loc[df.data_invalida | df.duracao_invalida], 'ocorrencias_revisao')
    df['ibge_ausente_ou_suspeito'] = ~df.CodIBGE.astype('string').str.fullmatch('41\\d{5}', na=False)
    for c in ['MdaPreparo', 'MdaDeslocamento', 'MdaExecucao']:
        v = pd.to_numeric(df[c], errors='coerce')
        df[c + '_valido'] = v.mask(v.lt(0))
        quality['ocorrencias'][c + '_nulos_ou_negativos'] = int((v.isna() | v.lt(0)).sum())
    quality['ocorrencias']['ibge_ausente_ou_suspeito'] = int(df.ibge_ausente_ou_suspeito.sum())
    key = ['NumCPFCNPJ', 'NumOcorrencia', 'DthInicioOcorrenciaAberta']
    repeated = df.duplicated(key, keep=False)
    quality['ocorrencias']['linhas_chave_repetida'] = int(repeated.sum())
    df = df.loc[~df.data_invalida].copy()
    save(df, 'ocorrencias_copel')
    return df.groupby('data').agg(n_registros_ocorrencia=('NumOcorrencia', 'size'), n_numeros_ocorrencia=('NumOcorrencia', 'nunique'))

In [8]:
def ons():
    frames = []
    for year in YEARS:
        p = RAW / f'ons_CURVA_CARGA_{year}.csv'
        d = pd.read_csv(p, sep=';')
        schemas[p.name] = {c: str(t) for (c, t) in d.dtypes.items()}
        require(d, ['id_subsistema', 'din_instante', 'val_cargaenergiahomwmed'], 'ons')
        frames.append(d.loc[d.id_subsistema.eq('S')].copy())
    df = pd.concat(frames, ignore_index=True)
    before = len(df)
    df = df.drop_duplicates()
    df['din_instante'] = pd.to_datetime(df.din_instante, errors='coerce')
    if df.duplicated(['id_subsistema', 'din_instante']).any():
        raise ValueError('ONS: timestamps conflitantes')
    df['carga_mwmed'] = pd.to_numeric(df.val_cargaenergiahomwmed, errors='coerce')
    bad = df.din_instante.isna() | df.carga_mwmed.isna() | df.carga_mwmed.lt(0)
    save(df.loc[bad], 'ons_revisao')
    quality['ons'] = {'originais_sul': before, 'duplicados_exatos': before - len(df), 'invalidos': int(bad.sum()), 'zeros': int(df.carga_mwmed.eq(0).sum())}
    df.loc[bad, 'carga_mwmed'] = float('nan')
    df['data'] = df.din_instante.dt.normalize()
    save(df, 'ons_sul_horario')
    g = df.groupby('data')
    daily = g.agg(ons_sul_carga_media_mwmed=('carga_mwmed', 'mean'), ons_sul_carga_max_horaria_mwmed=('carga_mwmed', 'max'), ons_sul_horas_validas=('carga_mwmed', 'count'))
    daily['ons_dia_completo'] = daily.ons_sul_horas_validas.eq(24)
    daily.loc[~daily.ons_dia_completo, ['ons_sul_carga_media_mwmed', 'ons_sul_carga_max_horaria_mwmed']] = float('nan')
    return daily

def clima():
    frames = []
    for name in PONTOS:
        d = json.loads((RAW / f'clima_{name}_2026.json').read_text())
        assert d['timezone'] == 'America/Sao_Paulo'
        df = pd.DataFrame(d['daily']).rename(columns={'time': 'data'})
        df['data'] = pd.to_datetime(df.data)
        df['ponto'] = name
        if df.duplicated(['data']).any():
            raise ValueError('Clima duplicado')
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    save(df, 'clima_pontos')
    cols = [c for c in df if c not in ['data', 'ponto']]
    g = df.groupby('data')
    daily = g[cols].mean().add_prefix('clima_media_')
    daily['clima_max_precipitacao_mm'] = g.precipitation_sum.max()
    daily['clima_max_rajada_kmh'] = g.wind_gusts_10m_max.max()
    daily['clima_pontos_completos'] = g[cols].count().min(axis=1)
    vals = [c for c in daily if c != 'clima_pontos_completos']
    daily.loc[daily.clima_pontos_completos.ne(5), vals] = float('nan')
    quality['clima'] = {'linhas_ponto_dia': len(df), 'nulos': df[cols].isna().sum().to_dict()}
    return daily

### Exportação e auditoria

O calendário diário reúne as fontes por data. A exportação mantém o Parquet final, os microdados filtrados e a auditoria no notebook.

In [9]:
def meaning(c):
    if c == 'data':
        return ('data civil local', 'data')
    if c == 'n_registros_interrupcao':
        return ('Registros ANEEL após duplicatas exatas; não são necessariamente defeitos físicos independentes', 'registros')
    if c == 'n_ids_interrupcao':
        return ('Identificadores distintos ano/ordem até 2025 e ano/CodInterrupcao em 2026, por dia de início', 'identificadores')
    if c == 'n_nao_programadas':
        return ('Registros classificados como Não Programada na fonte', 'registros')
    if c == 'n_duracoes_validas':
        return ('Registros com duração não nula e não negativa', 'registros')
    if c == 'duracao_media_h':
        return ('Média das durações válidas de eventos iniciados no dia', 'h')
    if c == 'uc_eventos_soma':
        return ('Soma de UCs afetadas por registro, com possíveis repetições de UC', 'UC-eventos')
    if c == 'uc_horas_soma':
        return ('Soma de UC-eventos multiplicados pela duração; atribuída ao início; não é DEC oficial', 'UC-eventos h')
    if c == 'n_registros_ocorrencia':
        return ('Registros de atendimento/ocorrência emergencial; não equivalem à contagem de interrupções', 'registros')
    if c == 'n_numeros_ocorrencia':
        return ('Números de ocorrência distintos por dia de abertura', 'identificadores')
    if c == 'ons_sul_horas_validas':
        return ('Número de valores horários válidos de carga do subsistema Sul', 'horas')
    if c == 'ons_dia_completo':
        return ('24 valores horários válidos no dia; ausente quando não há data na fonte', 'booleano')
    if c.startswith('ons_sul_carga_'):
        return ('Carga do subsistema Sul, agregada de CSV horário; não corresponde à demanda exclusiva do PR', 'MWmed')
    if c == 'clima_pontos_completos':
        return ('Menor contagem de pontos válidos entre as variáveis meteorológicas', 'pontos de 5')
    if c.startswith('clima_'):
        unit = '°C' if 'temperature' in c else 'km/h' if 'wind' in c or 'rajada' in c else 'mm/dia'
        return ('Média simples dos cinco pontos ERA5' if c.startswith('clima_media') else 'Máximo espacial entre os cinco pontos ERA5', unit)
    if c == 'feriado_nome':
        return ('Nome retornado por holidays BR/PR; nulo significa sem feriado no calendário', 'texto')
    if c == 'feriado_br_pr':
        return ('Data presente no calendário holidays BR/PR; não inclui regra própria para pontos facultativos', 'booleano')
    if c == 'dia_semana':
        return ('Segunda=0 a domingo=6', 'código')
    if c == 'fim_semana':
        return ('Sábado ou domingo', 'booleano')
    if c.startswith('sem_registro_'):
        return ('Não há registro da fonte neste dia; não se presume contagem zero', 'booleano')
    if c == 'aneel_esquema_2026':
        return ('Regime do esquema ANEEL 2026; indica mudança de campos/identificadores', 'booleano')
    if c == 'ons_pos_mmgd_20230429':
        return ('Marca de sensibilidade documental; não é correção numérica da carga', 'booleano')
    raise ValueError(c)

In [10]:
def main():
    base = pd.DataFrame(index=pd.date_range('2022-01-01', END, name='data'))
    for part in [interrupcoes(), ocorrencias(), ons(), clima()]:
        base = base.join(part, validate='one_to_one')
    br = holidays.country_holidays('BR', subdiv='PR', years=list(YEARS), language='pt_BR')
    cal = pd.DataFrame({'data': base.index})
    cal['feriado_nome'] = cal.data.map(lambda d: br.get(d.date()))
    cal['feriado_br_pr'] = cal.feriado_nome.notna()
    cal['dia_semana'] = cal.data.dt.dayofweek
    cal['fim_semana'] = cal.dia_semana.ge(5)
    base = base.join(cal.set_index('data'), validate='one_to_one')
    base['sem_registro_interrupcao'] = base.n_registros_interrupcao.isna()
    base['sem_registro_ocorrencia'] = base.n_registros_ocorrencia.isna()
    base['aneel_esquema_2026'] = base.index.year >= 2026
    base['ons_pos_mmgd_20230429'] = base.index >= pd.Timestamp('2023-04-29')
    quality['base'] = {'dias': len(base), 'inicio': str(base.index.min()), 'fim': str(base.index.max()), 'nulos_por_coluna': base.isna().sum().to_dict(), 'dias_sem_registro_interrupcao': int(base.sem_registro_interrupcao.sum())}
    assert base.index.is_unique and len(base) == len(pd.date_range('2022-01-01', END))
    save(base.reset_index(), 'base_diaria_copel_pr')
    manifest = []
    for p in sorted(RAW.iterdir()):
        if p.is_file() and p.suffix != '.part':
            h = hashlib.sha256()
            with p.open('rb') as f:
                for chunk in iter(lambda : f.read(1024 * 1024), b''):
                    h.update(chunk)
            manifest.append({'arquivo': p.name, 'bytes': p.stat().st_size, 'sha256': h.hexdigest()})
    manifesto = {'processado_em_utc': datetime.now(timezone.utc).isoformat(),
                 'python': platform.python_version(), 'pandas': pd.__version__,
                 'holidays': holidays.__version__, 'arquivos': manifest}
    (OUT / 'manifesto.json').write_text(json.dumps(manifesto, indent=2))
    return base.reset_index(), manifesto

quality.clear()
schemas.clear()
base, manifesto = main()
qualidade = quality
print(f'{len(base)} dias | {len(base.columns)} colunas')

1781 dias | 30 colunas


In [11]:
campos = ['linhas_copel_originais', 'duplicados_exatos', 'linhas_sem_duplicados', 'linhas_chave_repetida']
auditoria = pd.DataFrame({fonte: {c: qualidade[fonte][c] for c in campos}
                          for fonte in ['interrupcoes', 'ocorrencias']}).T
display(auditoria)

suspeitos = {
    fonte: {c: valor for c, valor in qualidade[fonte].items() if c not in campos}
    for fonte in ['interrupcoes', 'ocorrencias']
}
display(pd.DataFrame(suspeitos).T)
display(pd.DataFrame([qualidade['ons']], index=['ONS Sul']))
display(pd.Series(qualidade['clima']['nulos'], name='nulos_clima_por_variavel'))

,linhas_copel_originais,duplicados_exatos,linhas_sem_duplicados,linhas_chave_repetida
interrupcoes,3872436,685503,3186933,2396
ocorrencias,3460275,22352,3437923,230807


,inicio_invalido,duracao_invalida,uc_invalida,duracao_suspeita_7dias,data_invalida,MdaPreparo_nulos_ou_negativos,MdaDeslocamento_nulos_ou_negativos,MdaExecucao_nulos_ou_negativos,ibge_ausente_ou_suspeito
interrupcoes,0.0,0.0,0.0,910.0,NaN,NaN,NaN,NaN,NaN
ocorrencias,NaN,0.0,NaN,NaN,0.0,11204.0,179.0,9739.0,6159.0


,originais_sul,duplicados_exatos,invalidos,zeros
ONS Sul,42864,0,0,0


temperature_2m_mean    0
temperature_2m_max     0
temperature_2m_min     0
precipitation_sum      0
wind_gusts_10m_max     0
Name: nulos_clima_por_variavel, dtype: int64

In [12]:
def amostra(nome, colunas):
    arquivo = pq.ParquetFile(OUT / f'{nome}.parquet')
    lote = next(arquivo.iter_batches(batch_size=5, columns=colunas))
    return lote.to_pandas()

display(amostra('interrupcoes_copel', [
    'NumCPFCNPJ', 'data', 'id_interrupcao', 'DscTipoInterrupcao',
    'DscFatoGeradorInterrupcao', 'duracao_h_valida', 'esquema_aneel',
]))
display(amostra('ocorrencias_copel', [
    'NumCPFCNPJ', 'data', 'NumOcorrencia', 'duracao_atendimento_h', 'esquema_aneel',
]))

,NumCPFCNPJ,data,id_interrupcao,DscTipoInterrupcao,DscFatoGeradorInterrupcao,duracao_h_valida,esquema_aneel
0,04368898000106,2022-01-01,2022|2002014997-14593,Não Programada,INTERNA-NAO-PROGRAMADA-PROPRIAS DO SISTEMA-DESLIGAMENTO PARA ...,4.883333,ate2025
1,04368898000106,2022-01-01,2022|1021847626-14665,Não Programada,INTERNA-NAO-PROGRAMADA-PROPRIAS DO SISTEMA-NAO IDENTIFICADA,0.000000,ate2025
2,04368898000106,2022-01-01,2022|1021847625-16755,Não Programada,INTERNA-NAO-PROGRAMADA-PROPRIAS DO SISTEMA-NAO IDENTIFICADA,0.000000,ate2025
3,04368898000106,2022-01-01,2022|2031926597-15571,Não Programada,INTERNA-NAO-PROGRAMADA-TERCEIROS-OBJETO NA REDE,9.500000,ate2025
4,04368898000106,2022-01-01,2022|2042978617-15569,Não Programada,INTERNA-NAO-PROGRAMADA-PROPRIAS DO SISTEMA-DESLIGAMENTO PARA ...,0.033333,ate2025


,NumCPFCNPJ,data,NumOcorrencia,duracao_atendimento_h,esquema_aneel
0,04368898000106,2022-01-01,NRO - 2033480847,9.533333,ate2025
1,04368898000106,2022-01-01,NRO - 2033480847,9.533333,ate2025
2,04368898000106,2022-01-01,NRT - 1023559593,9.616667,ate2025
3,04368898000106,2022-01-01,NRT - 1023559593,9.616667,ate2025
4,04368898000106,2022-01-01,LES - 2004805482,1.883333,ate2025


## 4. Agregação e integração

A integração usa a data civil local. A contagem é atribuída ao início da interrupção ou da ocorrência. O identificador de interrupção combina ano e número de ordem até 2025; em 2026, combina ano e `CodInterrupcao`.

A carga diária usa média e máximo de 24 valores horários válidos. O clima usa média simples dos cinco pontos e máximos espaciais de chuva e rajada; exige todos os pontos completos. Os feriados vêm do calendário BR/PR da biblioteca `holidays`.

UC-eventos podem contar o mesmo consumidor mais de uma vez. Duração e UC-horas são atribuídas integralmente ao dia de início.

In [13]:
dia = base.loc[
    base.data.dt.year.eq(2026)
    & base.ons_dia_completo.eq(True)
    & base.clima_pontos_completos.eq(len(PONTOS)),
    'data',
].iloc[0]

horario = pd.read_parquet(OUT / 'ons_sul_horario.parquet')
carga_dia = horario.loc[horario.data.eq(dia), 'carga_mwmed']
clima = pd.read_parquet(OUT / 'clima_pontos.parquet')
clima_dia = clima.loc[clima.data.eq(dia)]
linha = base.loc[base.data.eq(dia)].iloc[0]

conferencia = pd.DataFrame([
    ['Carga média Sul (MWmed)', carga_dia.mean(), linha.ons_sul_carga_media_mwmed],
    ['Máximo horário Sul (MWmed)', carga_dia.max(), linha.ons_sul_carga_max_horaria_mwmed],
    ['Temperatura média dos pontos (°C)', clima_dia.temperature_2m_mean.mean(), linha.clima_media_temperature_2m_mean],
], columns=['medida', 'calculado_dos_dados', 'base_integrada'])
assert len(carga_dia) == 24
assert len(clima_dia) == len(PONTOS)
assert (conferencia.calculado_dos_dados - conferencia.base_integrada).abs().lt(1e-8).all()
print(f'Conferência: {dia.date()}')
display(conferencia)
display(clima_dia[['ponto', 'temperature_2m_mean', 'precipitation_sum', 'wind_gusts_10m_max']])

Conferência: 2026-01-01


,medida,calculado_dos_dados,base_integrada
0,Carga média Sul (MWmed),11203.195917,11203.195917
1,Máximo horário Sul (MWmed),13707.431000,13707.431000
2,Temperatura média dos pontos (°C),22.560000,22.560000


,ponto,temperature_2m_mean,precipitation_sum,wind_gusts_10m_max
1461,curitiba,22.3,10.7,60.8
3242,londrina,23.4,29.3,41.4
5023,maringa,23.7,22.6,44.3
6804,cascavel,21.9,5.6,31.7
8585,ponta_grossa,21.5,12.5,58.7


In [14]:
display(base.loc[
    base.data.dt.year.eq(2026) & base.feriado_br_pr,
    ['data', 'feriado_nome', 'dia_semana'],
].head(8))

display(base.loc[base.data.eq(dia), [
    'data', 'n_ids_interrupcao', 'n_registros_ocorrencia',
    'ons_sul_carga_media_mwmed', 'clima_media_temperature_2m_mean', 'feriado_br_pr',
]])

,data,feriado_nome,dia_semana
1461,2026-01-01,Confraternização Universal,3
1553,2026-04-03,Sexta-feira Santa,4
1571,2026-04-21,Tiradentes,1
1581,2026-05-01,Dia do Trabalhador,4
1710,2026-09-07,Independência do Brasil,0
1745,2026-10-12,Nossa Senhora Aparecida,0
1766,2026-11-02,Finados,0
1779,2026-11-15,Nossa Senhora do Rocio; Proclamação da República,6


,data,n_ids_interrupcao,n_registros_ocorrencia,ons_sul_carga_media_mwmed,clima_media_temperature_2m_mean,feriado_br_pr
1461,2026-01-01,3050.0,2603.0,11203.195917,22.56,True


## 5. Cobertura e ausências

Dia sem registro fica nulo, não zero. O calendário termina no corte da extração, mas cada fonte possui uma data de disponibilidade. A tabela mensal permite identificar lacunas dentro do período, inclusive em 2026.

In [15]:
medidas = {
    'Interrupções Copel': 'n_ids_interrupcao',
    'Ocorrências Copel': 'n_registros_ocorrencia',
    'Carga ONS Sul': 'ons_sul_carga_media_mwmed',
    'Clima dos cinco pontos': 'clima_media_temperature_2m_mean',
}
cobertura = []
for fonte, coluna in medidas.items():
    datas = base.loc[base[coluna].notna(), 'data']
    cobertura.append({
        'fonte': fonte,
        'primeiro_dia': datas.min().date(),
        'ultimo_dia': datas.max().date(),
        'dias_com_dado': len(datas),
        'dias_ausentes': int(base[coluna].isna().sum()),
    })
display(pd.DataFrame(cobertura))

mensal = base.assign(mes=base.data.dt.strftime('%Y-%m')).groupby('mes').agg(
    dias_calendario=('data', 'size'),
    dias_com_interrupcoes=('n_registros_interrupcao', 'count'),
    dias_com_ocorrencias=('n_registros_ocorrencia', 'count'),
    dias_com_ons=('ons_sul_carga_media_mwmed', 'count'),
    dias_com_clima=('clima_media_temperature_2m_mean', 'count'),
)
mensal['ano_esquema_2026'] = mensal.index.str.startswith('2026')
mensal.to_parquet(OUT / 'cobertura_mensal.parquet')
colunas = ['dias_com_interrupcoes', 'dias_com_ocorrencias', 'dias_com_ons', 'dias_com_clima']
lacuna = mensal[colunas].lt(mensal.dias_calendario, axis=0).any(axis=1)
display(mensal.loc[lacuna | mensal.index.str.startswith('2026'), ['dias_calendario', *colunas]])

,fonte,primeiro_dia,ultimo_dia,dias_com_dado,dias_ausentes
0,Interrupções Copel,2022-01-01,2026-09-30,1704,77
1,Ocorrências Copel,2022-01-01,2026-09-30,1716,65
2,Carga ONS Sul,2022-01-01,2026-11-16,1781,0
3,Clima dos cinco pontos,2022-01-01,2026-11-16,1781,0


,dias_calendario,dias_com_interrupcoes,dias_com_ocorrencias,dias_com_ons,dias_com_clima
mes,,,,,
2025-06,30,0,30,30,30
2026-01,31,31,31,31,31
2026-02,28,28,28,28,28
2026-03,31,31,31,31,31
2026-04,30,30,30,30,30
2026-05,31,31,31,31,31
2026-06,30,30,12,30,30
2026-07,31,31,31,31,31
2026-08,31,31,31,31,31


Junho de 2025 está sem registros de interrupção Copel no arquivo consultado; junho de 2026 tem cobertura parcial de ocorrências. O fim dos registros ANEEL em setembro de 2026 não informa o que aconteceu nos meses seguintes.

Comparações históricas precisam considerar cobertura e mudança de esquema. A recorrência e os fatores associados serão investigados na Etapa 3.

## 6. Parquet final e verificação

O resultado é `data/processed/base_diaria_copel_pr.parquet`. O dicionário registra as unidades e a interpretação das colunas; o manifesto local em `data/processed/manifesto.json` registra os hashes dos originais e aparece na saída da verificação.

In [16]:
dicionario = pd.DataFrame.from_dict({
    coluna: {'descricao': meaning(coluna)[0], 'tipo': str(base[coluna].dtype),
             'unidade': meaning(coluna)[1], 'nulos': int(base[coluna].isna().sum())}
    for coluna in base
}, orient='index')
with pd.option_context('display.max_colwidth', None):
    display(dicionario)

R = ROOT
P = OUT
b = pd.read_parquet(P / 'base_diaria_copel_pr.parquet')
assert len(b) == len(pd.date_range('2022-01-01', '2026-11-16')) and b.data.is_unique
assert b.data.tolist() == list(pd.date_range('2022-01-01', '2026-11-16'))
assert b.loc[b.sem_registro_interrupcao, 'n_registros_interrupcao'].isna().all()
assert b.loc[b.sem_registro_ocorrencia, 'n_registros_ocorrencia'].isna().all()
assert b.loc[~b.ons_dia_completo.astype('boolean').fillna(False), 'ons_sul_carga_media_mwmed'].isna().all()
i = pd.read_parquet(P / 'interrupcoes_copel.parquet', columns=['data', 'NumCPFCNPJ', 'duracao_h', 'duracao_h_valida', 'id_interrupcao'])
assert i.NumCPFCNPJ.eq('04368898000106').all()
assert i.loc[i.duracao_h.lt(0), 'duracao_h_valida'].isna().all()
count = i.groupby('data').size().rename('real').reset_index()
z = b.merge(count, on='data', how='left', validate='1:1')
assert z.n_registros_interrupcao.equals(z.real.astype('float64'))
assert (b.n_ids_interrupcao.dropna() <= b.n_registros_interrupcao.dropna()).all()
cl = pd.read_parquet(P / 'clima_pontos.parquet')
assert not cl.duplicated(['data', 'ponto']).any() and len(cl) == 5 * len(b)
assert b.loc[b.clima_pontos_completos.ne(5), 'clima_max_precipitacao_mm'].isna().all()
assert b.feriado_br_pr.dtype == bool
assert b.loc[b.data.dt.strftime('%m-%d').eq('01-01'), 'feriado_br_pr'].all()
manifest = json.loads((OUT / 'manifesto.json').read_text())
assert all(((R / 'data/raw' / x['arquivo']).stat().st_size == x['bytes'] for x in manifest['arquivos']))
print('OK: calendário contínuo, chaves, CNPJ, contagens reconciliadas, ausências, clima, feriados e manifesto.')

print(f"Parquet: {(OUT / 'base_diaria_copel_pr.parquet').stat().st_size / 1024:.1f} KiB")
print('Versões da execução:', {chave: manifesto[chave] for chave in ['python', 'pandas', 'holidays']})
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(manifesto['arquivos']))

,descricao,tipo,unidade,nulos
data,data civil local,datetime64[ns],data,0
n_registros_interrupcao,Registros ANEEL após duplicatas exatas; não são necessariamente defeitos físicos independentes,float64,registros,77
n_ids_interrupcao,"Identificadores distintos ano/ordem até 2025 e ano/CodInterrupcao em 2026, por dia de início",float64,identificadores,77
n_nao_programadas,Registros classificados como Não Programada na fonte,float64,registros,77
n_duracoes_validas,Registros com duração não nula e não negativa,float64,registros,77
duracao_media_h,Média das durações válidas de eventos iniciados no dia,float64,h,77
uc_eventos_soma,"Soma de UCs afetadas por registro, com possíveis repetições de UC",Int64,UC-eventos,77
uc_horas_soma,Soma de UC-eventos multiplicados pela duração; atribuída ao início; não é DEC oficial,Float64,UC-eventos h,77
n_registros_ocorrencia,Registros de atendimento/ocorrência emergencial; não equivalem à contagem de interrupções,float64,registros,65
n_numeros_ocorrencia,Números de ocorrência distintos por dia de abertura,float64,identificadores,65


OK: calendário contínuo, chaves, CNPJ, contagens reconciliadas, ausências, clima, feriados e manifesto.
Parquet: 179.5 KiB
Versões da execução: {'python': '3.10.12', 'pandas': '2.3.3', 'holidays': '0.107'}


,arquivo,bytes,sha256
0,.gitkeep,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
1,clima_cascavel_2026.json,141645,710f4c80941d1b671d41c5a3905e5bbde13127998558e5321245d7a9e0ff0328
2,clima_curitiba_2026.json,141581,91b7e234d86dd6e88700b98396eed1f536eb37c9dd45857e848b4ad5d51e922e
3,clima_londrina_2026.json,141797,1ec77cf95b8e408c7deea0387dae5061948d2a66f9489e2c1ea2f0cd942b8418
4,clima_maringa_2026.json,141764,2871bb5c04185a5be47b19b6d2df3a0abd8b129b6357eff4d827dc028547a22e
5,clima_ponta_grossa_2026.json,141568,78db4831984426f01d9daf5e9c69a75babe1002a1b5b57690b03a1adc651a949
6,interrupcoes_catalogo.json,33969,2d4f83f048bfb9a5e4508877f51f9b563916f60521edc4015ee746f641420436
7,interrupcoes_dm-interrupcoes-de-energia-eletrica-nas-redes-de-distribuicao.pdf,165260,e7234c20b9f8d478ffea7d83c121de8811e93a869e13e6ced38ed58b26fe566c
8,interrupcoes_interrupcoes-energia-eletrica-2022.parquet,229322133,c02d7752a62cf3a6d1eb687e1cfcd3c15300afd0f2aeb5c3562a62e34b6c1141
9,interrupcoes_interrupcoes-energia-eletrica-2023.parquet,246009645,fc9f9413fd440d5e50ab477ec18cf8feafd083197785d63859f629379d051d6e


Para D+1, os valores observados no dia-alvo não podem entrar como informações já conhecidas. A modelagem também deve considerar o atraso de publicação das fontes. Esta base é retrospectiva; associações não demonstram causalidade.